In [1]:
!pip install openai-whisper
!pip install pybind11 


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 13.3 MB/s eta 0:00:00a 0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803979 sha256=69040c7dd0a4a83458381903536d990f241b66b655d29df249a9649057f354c9
  Stored in directory: /root/.cache/pip/wheels/61/d2/20/09ec9bef734d126cba375b15898010b6cc28578d8afdde5869
Successfully built openai-whisper


In [2]:
# 2. Fetch the extension dynamically using pure Python instead of python3-config
!export EXT=$(python3 -c "import sysconfig; print(sysconfig.get_config_var('EXT_SUFFIX'))") && \
 export SRC="/kaggle/input/datasets/aneeshshastri/custom-tokenizers/bpe_tokenizer.cpp" && \
 g++ -O3 -Wall -shared -std=c++17 -fPIC $(python3 -m pybind11 --includes) $SRC -o /kaggle/working/bpe_tokenizer$EXT

# 3. Verify compilation success
!ls -l /kaggle/working/ | grep bpe_tokenizer

-rwxr-xr-x 1 root root 314544 May 10 10:37 bpe_tokenizer.cpython-312-x86_64-linux-gnu.so


In [3]:
import jax
import jax.numpy as jnp
import flax.nnx as nnx
import optax
import grain
import pathlib
import librosa
import numpy as np
import optax
import bpe_tokenizer
from tqdm import tqdm
import math
import collections
import itertools
import whisper
from pathlib import Path
from sklearn.model_selection import train_test_split
RANDOM_SEED=42

In [12]:
EMOTION_MAP = {
    '01': 0,  # neutral
    '02': 1,  # calm
    '03': 2,  # happy
    '04': 3,  # sad
    '05': 4,  # angry
    '06': 5,  # fearful
    '07': 6,  # disgust
    '08': 7,  # surprised
}

def parse_ravdess(root: str) -> list[dict]:
    records = []
    for path in sorted(pathlib.Path(root).rglob("*.wav")):
        parts = path.stem.split('-')
        records.append({
            "path": str(path),
            "path"
            "label": EMOTION_MAP[parts[2]],
            "actor": int(parts[6]),
            "intensity": int(parts[3]),
        })
    return records

In [5]:

def transcribe_simple_corpus(input_path: str, output_path: str, model_size: str = "base") -> str:
    in_dir = Path(input_path)
    out_dir = Path(output_path)
    
    if not in_dir.exists() or not in_dir.is_dir():
        raise FileNotFoundError(f"Dataset path does not exist: {input_path}")

    # Create the writable output directory if it doesn't exist
    out_dir.mkdir(parents=True, exist_ok=True)

    wav_files = list(in_dir.rglob("*.wav"))
    if not wav_files:
        raise ValueError(f"No .wav files found in {input_path}")
        
    print(f"Located {len(wav_files)} .wav files. Loading Whisper '{model_size}'...")
    
    model = whisper.load_model(model_size)
    corpus_texts = []

    for audio_path in tqdm(wav_files, desc="Transcribing RAVDESS"):
        result = model.transcribe(
            str(audio_path),
            temperature=0.0,
            condition_on_previous_text=False,
            no_speech_threshold=0.6
        )
        
        transcription = result["text"].strip()
        
        # Route the output strictly to the writable directory using the unique stem
        txt_file_name = f"{audio_path.stem}.txt"
        txt_path = out_dir / txt_file_name
        
        with open(txt_path, "w", encoding="utf-8") as f:
            f.write(transcription)
            
        corpus_texts.append(transcription)

    return " ".join(corpus_texts)

# ==========================================
# Execution
# ==========================================
if __name__ == "__main__":
    # The read-only Kaggle mount
    RAVDESS_INPUT = "/kaggle/input/datasets/uwrfkaggler/ravdess-emotional-speech-audio"
    
    # The writable working directory
    TRANSCRIPTS_OUTPUT = "/kaggle/working/ravdess_transcripts"
    
    full_corpus_string = transcribe_simple_corpus(
        input_path=RAVDESS_INPUT, 
        output_path=TRANSCRIPTS_OUTPUT, 
        model_size="base"
    )
    
    print("\nTranscription complete.")
    print(f"Transcripts saved to: {TRANSCRIPTS_OUTPUT}")
    print(f"Total characters in corpus: {len(full_corpus_string)}")

Located 2880 .wav files. Loading Whisper 'base'...


100%|████████████████████████████████████████| 139M/139M [00:00<00:00, 239MiB/s]
Transcribing RAVDESS: 100%|██████████| 2880/2880 [14:32<00:00,  3.30it/s]


Transcription complete.
Transcripts saved to: /kaggle/working/ravdess_transcripts
Total characters in corpus: 86695


In [6]:
Tokenizer=bpe_tokenizer.BPETokenizer()
Tokenizer.train(full_corpus_string,num_merges=1_000)
print(Tokenizer.get_vocab_size())

447


In [7]:

MEL_CFG = dict(
    sr          = 22050,
    n_fft       = 1024,
    hop_length  = 256,       # ~11.6 ms stride at 22 kHz
    n_mels      = 128,
    fmin        = 50,
    fmax        = 8000,
    duration    = 3.0,       # the clip lengths are all exactly 3 seconds
)

def load_melspec(path: str, cfg: dict = MEL_CFG) -> np.ndarray:
    wav, _ = librosa.load(path, sr=cfg["sr"], mono=True)
    target_len = int(cfg["sr"] * cfg["duration"])
    
    if len(wav) < target_len:
        wav = np.pad(wav, (0, target_len - len(wav)))
    else:
        wav = wav[:target_len]
    # not necessary, but just in case

    # 3. Mel spectrogram → log scale
    mel = librosa.feature.melspectrogram(
        y=wav,
        sr=cfg["sr"],
        n_fft=cfg["n_fft"],
        hop_length=cfg["hop_length"],
        n_mels=cfg["n_mels"],
        fmin=cfg["fmin"],
        fmax=cfg["fmax"],
    )
    log_mel = librosa.power_to_db(mel, ref=np.max)  # shape: (n_mels, T)
    # 4. Normalize to [0, 1] per sample (or use dataset-level stats)
    log_mel = (log_mel - log_mel.min()) / (log_mel.max() - log_mel.min() + 1e-8)
    # 5. Add channel dim → (H, W, C) for CNNs: (128, T, 1)
    return log_mel[..., np.newaxis].astype(np.float32)

In [13]:

def preprocess_and_cache(records: list[dict], cache_dir: str):
    cache = pathlib.Path(cache_dir)
    cache.mkdir(exist_ok=True)
    
    specs, labels = [], []
    for rec in tqdm(records):
        npy_path = cache / (pathlib.Path(rec["path"]).stem + ".npy")
        if not npy_path.exists():
            mel = load_melspec(rec["path"])
            np.save(npy_path, mel)
        else:
            mel = np.load(npy_path)
        specs.append(mel)
        labels.append(rec["label"])
    
    # Save as memory-mapped arrays for fast Grain access
    X = np.stack(specs)  
    y = np.array(labels, dtype=np.int32)
    np.save(cache / "X.npy", X)
    np.save(cache / "y.npy", y)
    return X, y

In [10]:


def preprocess_transcription(tokenizer, root_dir: str):
    """
    Loads text files, encodes them using the BPE tokenizer, 
    and pads them into a static-shaped JAX/NumPy compatible matrix.
    """
    root_path = Path(root_dir)
    
    if not root_path.exists() or not root_path.is_dir():
        raise FileNotFoundError(f"Transcript directory not found: {root_dir}")

    txt_files = sorted(list(root_path.rglob("*.txt")))
    
    if not txt_files:
        raise ValueError(f"No .txt files found in {root_dir}")
        
    print(f"Discovered {len(txt_files)} transcript files. Encoding...")

    encoded_sequences = []

    # 1. Read and Encode
    for txt_path in tqdm(txt_files, desc="Encoding Text"):
        with open(txt_path, "r", encoding="utf-8") as f:
            text = f.read().strip()
            
        # The tokenizer will handle empty strings (from dropped files) natively,
        # returning an empty list [] which will become pure padding.
        encoded_tokens = tokenizer.encode(text)
        encoded_sequences.append(encoded_tokens)

    # 2. Determine Static Shape for XLA
    max_len = max(len(seq) for seq in encoded_sequences)
    print(f"Max sequence length detected: {max_len} tokens")
    PAD_ID = tokenizer.get_vocab_size()    
    # 3. Apply Padding
    # We use 0 for padding since our base vocab starts at 0, but the neural net's 
    # embedding layer will map 0 to a specific vector.
    num_samples = len(encoded_sequences)
    X_transcript = np.full((num_samples, max_len), PAD_ID, dtype=np.int32)

    for i, seq in enumerate(encoded_sequences):
        # Insert the actual tokens; the remainder of the row remains 0
        X_transcript[i, :len(seq)] = seq

    return X_transcript, max_len


    
TRANSCRIPTS_DIR = "/kaggle/working/ravdess_transcripts"
X_text, max_sequence_length = preprocess_transcription(Tokenizer, TRANSCRIPTS_DIR)
        
print("\nPreprocessing successful.")
print(f"X_transcript shape: {X_text.shape} (Samples x Timesteps)")
print(f"Data type: {X_text.dtype}")
print(f"Sample 0 encoded array: {X_text[0]}")


Discovered 1440 transcript files. Encoding...


Encoding Text: 100%|██████████| 1440/1440 [00:00<00:00, 5207.44it/s]

Max sequence length detected: 12 tokens

Preprocessing successful.
X_transcript shape: (1440, 12) (Samples x Timesteps)
Data type: int32
Sample 0 encoded array: [446 269 276 266 260 264  46 447 447 447 447 447]


In [14]:
records=parse_ravdess("/kaggle/input/datasets/uwrfkaggler/ravdess-emotional-speech-audio")
X,y=preprocess_and_cache(records,"/kaggle/working/cache")

100%|██████████| 2880/2880 [00:20<00:00, 138.82it/s] 


In [15]:

# Load cached arrays
X = np.load("cache/X.npy")   # (N, 128, 259, 1)
y = np.load("cache/y.npy")   # (N,)


In [23]:
import shutil
import pathlib

# 1. Delete the corrupted 2880-sample cache
shutil.rmtree("/kaggle/working/cache", ignore_errors=True)

# 2. The Fixed Parsing Function (Deduplication + Syntax Fix)
def parse_ravdess_dedup(root: str) -> list[dict]:
    records = []
    seen_stems = set()
    
    for path in sorted(pathlib.Path(root).rglob("*.wav")):
        # Strict Deduplication: If we already processed this exact stem, skip it.
        if path.stem in seen_stems:
            continue
            
        seen_stems.add(path.stem)
        parts = path.stem.split('-')
        
        records.append({
            "path": str(path),
            "label": EMOTION_MAP[parts[2]],
            "actor": int(parts[6]),
            "intensity": int(parts[3]),
        })
        
    return records

# 3. Rebuild the Audio Cache
records = parse_ravdess_dedup("/kaggle/input/datasets/uwrfkaggler/ravdess-emotional-speech-audio")
print(f"Unique audio files found: {len(records)} (Should be exactly 1440)")

X, y = preprocess_and_cache(records, "/kaggle/working/cache")
print(f"New aligned X shape: {X.shape}")
print(f"New aligned y shape: {y.shape}")

Unique audio files found: 1440 (Should be exactly 1440)


100%|██████████| 1440/1440 [00:19<00:00, 75.49it/s]


New aligned X shape: (1440, 128, 259, 1)
New aligned y shape: (1440,)


In [25]:
from sklearn.model_selection import train_test_split

# 1. First Split: Separate Test set (20%)
# Pass both X (audio) and X_text (transcripts) into the same call
X_tv, X_test, X_text_tv, X_text_test, y_tv, y_test = train_test_split(
    X, X_text, y, test_size=0.2, random_state=42
)

# 2. Second Split: Separate Validation set (10% of the remaining 80%)
X_train, X_valid, X_text_train, X_text_valid, y_train, y_valid = train_test_split(
    X_tv, X_text_tv, y_tv, test_size=0.1, random_state=42
)

# 3. One-way transfer to GPU VRAM
# Audio Data
X_train_vram = jax.device_put(X_train)
X_val_vram = jax.device_put(X_valid)
X_test_vram = jax.device_put(X_test)

# Text Data (The new BPE-encoded token matrices)
X_text_train_vram = jax.device_put(X_text_train)
X_text_val_vram = jax.device_put(X_text_valid)
X_text_test_vram = jax.device_put(X_text_test)

# Labels
y_train_vram = jax.device_put(y_train)
y_val_vram = jax.device_put(y_valid)
y_test_vram = jax.device_put(y_test)

# Print shapes to verify alignment
print(f"Train shapes: Audio {X_train_vram.shape}, Text {X_text_train_vram.shape}, Labels {y_train_vram.shape}")

Train shapes: Audio (1036, 128, 259, 1), Text (1036, 12), Labels (1036,)


In [26]:


class OptimizedSpecAugment(nnx.Module):
    def __init__(self, freq_mask_param: int, time_mask_param: int, rngs: nnx.Rngs):
        """
        freq_mask_param: Maximum width of the frequency mask (F in the paper)
        time_mask_param: Maximum width of the time mask (T in the paper)
        """
        self.freq_mask_param = freq_mask_param
        self.time_mask_param = time_mask_param
        
        # In NNX, we grab a specific PRNG stream for augmentation
        self.rng = rngs.augmentation() 

    def __call__(self, x: jnp.ndarray, training: bool) -> jnp.ndarray:
        # If we aren't training, just return the data unchanged
        if not training:
            return x

        B, T_dim, F_dim = x.shape
        
        # 1. Generate random parameters for the masks
        # f: The width of the mask (0 to freq_mask_param)
        # f0: The starting index (0 to F_dim - f)
        key_f, key_f0, key_t, key_t0 = jax.random.split(self.rng, 4)
        
        f = jax.random.randint(key_f, shape=(B, 1, 1), minval=0, maxval=self.freq_mask_param)
        f0 = jax.random.randint(key_f0, shape=(B, 1, 1), minval=0, maxval=F_dim)
        
        t = jax.random.randint(key_t, shape=(B, 1, 1), minval=0, maxval=self.time_mask_param)
        t0 = jax.random.randint(key_t0, shape=(B, 1, 1), minval=0, maxval=T_dim)

        # 2. Create index arrays (static shape, highly optimized by XLA)
        freq_indices = jnp.arange(F_dim).reshape(1, 1, F_dim)
        time_indices = jnp.arange(T_dim).reshape(1, T_dim, 1)

        # 3. Generate boolean masks using broadcasted comparisons
        # True if index is OUTSIDE the masked region, False if INSIDE
        # This completely avoids dynamic slicing
        freq_mask = (freq_indices < f0) | (freq_indices >= f0 + f)
        time_mask = (time_indices < t0) | (time_indices >= t0 + t)

        # 4. Apply the masks (element-wise multiplication)
        # XLA will fuse these operations into a single kernel
        x = jnp.where(freq_mask, x, 0.0)
        x = jnp.where(time_mask, x, 0.0)

        return x

In [27]:


# ── 1. The 1D Convolutional Block ─────────────────────────────────────────────

class Conv1DBlock(nnx.Module):
    """1D Conv -> BatchNorm -> GELU -> MaxPool1D"""
    def __init__(self, in_features: int, out_features: int, rngs: nnx.Rngs):
        # We use Conv (which acts as Conv1D when given a 1D kernel)
        self.conv = nnx.Conv(
            in_features=in_features,
            out_features=out_features,
            kernel_size=(3,),       # 1D kernel sliding over time
            padding="SAME",
            rngs=rngs,
        )
        self.bn = nnx.BatchNorm(num_features=out_features, rngs=rngs)

    def __call__(self, x: jnp.ndarray, training: bool) -> jnp.ndarray:
        x = self.conv(x)
        x = self.bn(x, use_running_average=not training)
        x = jax.nn.gelu(x)
        
        # MaxPool1D: Pool over time by a factor of 2 to downsample the sequence length
        x = jax.lax.reduce_window(
            x, -jnp.inf, jax.lax.max,
            window_dimensions=(1, 2, 1), # (batch, time, features)
            window_strides=(1, 2, 1),
            padding="VALID",
        )
        return x

# ── 2. The 1D CNN Backbone ────────────────────────────────────────────────────

class CNN1DBackbone(nnx.Module):
    def __init__(self, rngs: nnx.Rngs):
        # Input shape expected: (Batch, Time, Channels)
        # For Mel: Channels = 128 (the frequency bins)
        self.block1 = Conv1DBlock(in_features=128, out_features=64, rngs=rngs)
        self.block2 = Conv1DBlock(in_features=64,  out_features=128, rngs=rngs)
        self.block3 = Conv1DBlock(in_features=128, out_features=256, rngs=rngs)

    def __call__(self, x: jnp.ndarray, training: bool) -> jnp.ndarray:
        # Input: (B, Time=259, Freq=128)
        x = self.block1(x, training)  # -> (B, 129, 64)
        x = self.block2(x, training)  # -> (B, 64, 128)
        x = self.block3(x, training)  # -> (B, 32, 256)
        return x

# ── 3. The Full 1D-CRNN Model ─────────────────────────────────────────────────

class GRUCell(nnx.Module):
    def __init__(self, input_size: int, hidden_size: int, rngs: nnx.Rngs):
        self.Wr = nnx.Linear(input_size + hidden_size, hidden_size, rngs=rngs)
        self.Wz = nnx.Linear(input_size + hidden_size, hidden_size, rngs=rngs)
        self.Wh = nnx.Linear(input_size + hidden_size, hidden_size, rngs=rngs)
        self.hidden_size = hidden_size

    def __call__(self, x: jnp.ndarray, h: jnp.ndarray) -> jnp.ndarray:
        xh = jnp.concatenate([x, h], axis=-1)
        r  = jax.nn.sigmoid(self.Wr(xh))
        z  = jax.nn.sigmoid(self.Wz(xh))
        xh_reset = jnp.concatenate([x, r * h], axis=-1)
        h_cand = jnp.tanh(self.Wh(xh_reset))
        return (1 - z) * h + z * h_cand

class EmotionCRNN(nnx.Module):
    """
    1D CNN Backbone -> GRU -> Classifier
    """
    def __init__(self, num_classes: int = 8, gru_hidden: int = 128, rngs: nnx.Rngs = None):
        self.spec_augment = OptimizedSpecAugment(
            freq_mask_param=20, time_mask_param=30, rngs=rngs
        )
        self.cnn = CNN1DBackbone(rngs)
        #self.gru = GRUCell(input_size=256, hidden_size=gru_hidden, rngs=rngs)
        self.connect=nnx.Linear(256,gru_hidden,rngs=rngs)
        self.dropout = nnx.Dropout(rate=0.3, rngs=rngs)
        self.fc1 = nnx.Linear(gru_hidden, 64, rngs=rngs)
        self.fc2 = nnx.Linear(64, num_classes, rngs=rngs)
        self.gru_hidden = gru_hidden

    def __call__(self, x: jnp.ndarray, training: bool = True) -> jnp.ndarray:
        # 1. Ensure input is (Batch, Time, Features)
        # Your previous pipeline yielded (B, 128, 259, 1). 
        # We must reshape/transpose this to (B, Time=259, Freq=128) before the CNN.
        x = jnp.squeeze(x, axis=-1)       # Remove the dummy channel dim: (B, 128, 259)
        x = jnp.transpose(x, (0, 2, 1))   # Swap axes to get: (B, 259, 128)
        x = self.spec_augment(x, training=training)
        # 2. 1D CNN Feature Extraction
        x = self.cnn(x, training)         # Output: (B, Time=32, Features=256)

        # 3. GRU Sequence Modeling
        """
        B = x.shape[0]
        h = jnp.zeros((B, self.gru_hidden))

        def gru_step(h, x_t):
            h_new = self.gru(x_t, h)
            return h_new, h_new

        # Scan across the time dimension (axis 1)
        h, _ = jax.lax.scan(gru_step, h, jnp.transpose(x, (1, 0, 2)))
        
        # h is the final hidden state after processing the whole sequence
        """
        x = jnp.max(x, axis=1)
        x=jax.nn.gelu(self.connect(x))
        # 4. Classification
        x = self.dropout(x, deterministic=not training)
        x = jax.nn.gelu(self.fc1(x))
        return self.fc2(x)

In [28]:
import jax
import jax.numpy as jnp
from flax import nnx
import optax
import math
import numpy as np

class TextRNN(nnx.Module):
    def __init__(self, vocab_size: int,pad_id: int, embed_dim: int, hidden_dim: int, num_classes: int, rngs: nnx.Rngs):
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim
        self.hidden_dim = hidden_dim
        self.pad_id=pad_id
        # 1. Layers
        self.embed = nnx.Embed(num_embeddings=vocab_size+1, features=embed_dim, rngs=rngs)
        self.cell = nnx.GRUCell(in_features=embed_dim, hidden_features=hidden_dim, rngs=rngs)
        self.rnn = nnx.RNN(self.cell)
        self.dropout = nnx.Dropout(rate=0.3, rngs=rngs)
        self.classifier = nnx.Linear(in_features=hidden_dim, out_features=num_classes, rngs=rngs)

    def __call__(self, x: jax.Array, training: bool = True) -> jax.Array:
        # x shape: (batch_size, max_len)
        
        # 2. Isolate Valid Tokens (0 is padding)
        mask = (x != self.pad_id) 

        # 3. Embedding & Recurrence
        emb = self.embed(x) 
        rnn_output = self.rnn(emb) 

        # 4. Masked Mean Pooling
        mask_expanded = jnp.expand_dims(mask, axis=-1) 
        masked_output = rnn_output * mask_expanded

        sum_hidden = jnp.sum(masked_output, axis=1)
        valid_lengths = jnp.sum(mask, axis=1, keepdims=True)
        valid_lengths = jnp.maximum(valid_lengths, 1) # Prevent division by zero

        final_representation = sum_hidden / valid_lengths

        # 5. Classification Head
        x_drop = self.dropout(final_representation, deterministic=not training)
        logits = self.classifier(x_drop)
        
        return logits



In [34]:

def create_text_model_and_optimizer(vocab_size: int,pad_id: int, learning_rate: float = 3e-4):
    rngs = nnx.Rngs(params=0, dropout=1)
    model = TextRNN(
        vocab_size=vocab_size, 
        embed_dim=64, 
        hidden_dim=128, 
        num_classes=8, 
        pad_id=pad_id,       
        rngs=rngs
    )

    schedule = optax.warmup_cosine_decay_schedule(
        init_value=0.0,
        peak_value=learning_rate,
        warmup_steps=100,
        decay_steps=2000,
        end_value=1e-6,
    )
    optimizer = nnx.Optimizer(model, optax.adamw(schedule, weight_decay=1e-4), wrt=nnx.Param)
    return model, optimizer

def create_model_and_optimizer(learning_rate: float = 3e-4):
    rngs  = nnx.Rngs(params=0, dropout=1, batch_stats=2, augmentation=3)
    model = EmotionCRNN(num_classes=8, gru_hidden=128, rngs=rngs)

    # Cosine decay with linear warmup — important for small datasets
    schedule = optax.warmup_cosine_decay_schedule(
        init_value=0.0,
        peak_value=learning_rate,
        warmup_steps=200,
        decay_steps=5000,
        end_value=1e-6,
    )
    optimizer = nnx.Optimizer(model, optax.adamw(schedule, weight_decay=1e-4),wrt=nnx.Param)
    return model, optimizer


@nnx.jit
def train_step(model, optimizer,metrics, batch):
    def loss_fn(model):
        logits = model(batch["mel"], training=True)
        loss   = optax.softmax_cross_entropy_with_integer_labels(
            logits, batch["label"]
        ).mean()
        return loss, logits

    (loss, logits), grads = nnx.value_and_grad(loss_fn, has_aux=True)(model)
    optimizer.update(model,grads)
    acc = (jnp.argmax(logits, -1) == batch["label"]).mean()
    metrics.update(loss=loss, logits=logits, labels=batch['label'])
    return loss, acc


@nnx.jit
def eval_step(model, metrics, batch):
    logits = model(batch["mel"], training=False)
    loss   = optax.softmax_cross_entropy_with_integer_labels(
        logits, batch["label"]
    ).mean()
    acc = (jnp.argmax(logits, -1) == batch["label"]).mean()
    metrics.update(loss=loss, logits=logits, labels=batch['label'])
    return loss, acc

@nnx.jit
def text_train_step(model, optimizer, metrics, batch):
    def loss_fn(model):
        logits = model(batch["text"], training=True)
        loss = optax.softmax_cross_entropy_with_integer_labels(logits, batch["label"]).mean()
        return loss, logits

    (loss, logits), grads = nnx.value_and_grad(loss_fn, has_aux=True)(model)
    optimizer.update(model, grads)
    metrics.update(loss=loss, logits=logits, labels=batch['label'])
    return loss

@nnx.jit
def text_eval_step(model, metrics, batch):
    logits = model(batch["text"], training=False)
    loss = optax.softmax_cross_entropy_with_integer_labels(logits, batch["label"]).mean()
    metrics.update(loss=loss, logits=logits, labels=batch['label'])
    return loss



In [30]:

def get_vram_batches(X_audio, X_text, y, batch_size=32, shuffle=True, seed=42):
    """
    Yields strictly aligned, multimodal batches directly from VRAM.
    """
    num_samples = len(y)
    indices = np.arange(num_samples)
    
    if shuffle:
        rng = np.random.default_rng(seed)
        rng.shuffle(indices)
        
    num_batches = math.floor(num_samples / batch_size)
    
    for i in range(num_batches):
        batch_indices = indices[i * batch_size : (i + 1) * batch_size]
        
        # Slicing from JAX arrays returns device arrays natively
        yield {
            "mel": X_audio[batch_indices], 
            "text": X_text[batch_indices],
            "label": y[batch_indices]
        }

In [32]:
print("Starting zero-latency training loop...")
EPOCHS = 100
# Metrics (Tracking 'loss' correctly)
metrics = nnx.MultiMetric(
    loss=nnx.metrics.Average('loss'),  
    accuracy=nnx.metrics.Accuracy()    
)
val_metrics = nnx.MultiMetric(
    loss=nnx.metrics.Average('loss'),  
    accuracy=nnx.metrics.Accuracy()    
)

# Instantiate your new 1D-CRNN
model, optimizer = create_model_and_optimizer()

# Split the dataset (Assume you've done this step before loading to VRAM)
# For this example, we assume X_train_vram, y_train_vram, X_val_vram, y_val_vram exist.

for epoch in range(EPOCHS):
    
    # --- TRAINING PHASE ---
    model.train()
    # Notice we don't use prefetchers or Grain loaders anymore
    train_batches = get_vram_batches(
        X_train_vram,X_text_train_vram, y_train_vram, batch_size=32, shuffle=True, seed=epoch
    )
    
    for step, batch in enumerate(train_batches):
        train_step(model, optimizer, metrics, batch)
        
    # --- VALIDATION PHASE ---
    model.eval()
    val_batches = get_vram_batches(
        X_val_vram,X_text_val_vram, y_val_vram, batch_size=32, shuffle=False
    )
    
    for step, batch in enumerate(val_batches):
        eval_step(model, val_metrics, batch)
        
    print(f"Epoch {epoch} | Train: {metrics.compute()} | Val: {val_metrics.compute()}")
    
    # Reset metrics
    metrics.reset()
    val_metrics.reset()

Starting zero-latency training loop...
Epoch 0 | Train: {'loss': Array(2.366035, dtype=float32), 'accuracy': Array(0.12988281, dtype=float32)} | Val: {'loss': Array(2.0807543, dtype=float32), 'accuracy': Array(0.14583334, dtype=float32)}
Epoch 1 | Train: {'loss': Array(2.2055066, dtype=float32), 'accuracy': Array(0.16015625, dtype=float32)} | Val: {'loss': Array(2.079564, dtype=float32), 'accuracy': Array(0.14583334, dtype=float32)}
Epoch 2 | Train: {'loss': Array(2.1074398, dtype=float32), 'accuracy': Array(0.1640625, dtype=float32)} | Val: {'loss': Array(2.0768967, dtype=float32), 'accuracy': Array(0.14583334, dtype=float32)}
Epoch 3 | Train: {'loss': Array(2.0651093, dtype=float32), 'accuracy': Array(0.18359375, dtype=float32)} | Val: {'loss': Array(2.0693748, dtype=float32), 'accuracy': Array(0.14583334, dtype=float32)}
Epoch 4 | Train: {'loss': Array(1.980724, dtype=float32), 'accuracy': Array(0.20996094, dtype=float32)} | Val: {'loss': Array(2.0602412, dtype=float32), 'accuracy':

In [36]:
print("Starting zero-latency training loop...")
VOCAB_SIZE = Tokenizer.get_vocab_size() 
EPOCHS = 50
BATCH_SIZE = 32
PAD_ID=VOCAB_SIZE
# Metrics (Tracking 'loss' correctly)
metrics = nnx.MultiMetric(
    loss=nnx.metrics.Average('loss'),  
    accuracy=nnx.metrics.Accuracy()    
)
val_metrics = nnx.MultiMetric(
    loss=nnx.metrics.Average('loss'),  
    accuracy=nnx.metrics.Accuracy()    
)

txt_model,txt_optimizer = create_text_model_and_optimizer(vocab_size=VOCAB_SIZE,pad_id=PAD_ID)

for epoch in range(EPOCHS):
    
    # --- TRAINING PHASE ---
    txt_model.train()
    # Notice we don't use prefetchers or Grain loaders anymore
    train_batches = get_vram_batches(
        X_train_vram,X_text_train_vram, y_train_vram, batch_size=32, shuffle=True, seed=epoch
    )
    
    for step, batch in enumerate(train_batches):
        text_train_step(txt_model,txt_optimizer, metrics, batch)
        
    # --- VALIDATION PHASE ---
    txt_model.eval()
    val_batches = get_vram_batches(
        X_val_vram,X_text_val_vram, y_val_vram, batch_size=32, shuffle=False
    )
    
    for step, batch in enumerate(val_batches):
        text_eval_step(txt_model, val_metrics, batch)
        
    print(f"Epoch {epoch} | Train: {metrics.compute()} | Val: {val_metrics.compute()}")
    
    # Reset metrics
    metrics.reset()
    val_metrics.reset()

Starting zero-latency training loop...
Epoch 0 | Train: {'loss': Array(2.0760126, dtype=float32), 'accuracy': Array(0.14550781, dtype=float32)} | Val: {'loss': Array(2.073576, dtype=float32), 'accuracy': Array(0.14583334, dtype=float32)}
Epoch 1 | Train: {'loss': Array(2.0758283, dtype=float32), 'accuracy': Array(0.13378906, dtype=float32)} | Val: {'loss': Array(2.071506, dtype=float32), 'accuracy': Array(0.16666667, dtype=float32)}
Epoch 2 | Train: {'loss': Array(2.0705867, dtype=float32), 'accuracy': Array(0.13476562, dtype=float32)} | Val: {'loss': Array(2.0711026, dtype=float32), 'accuracy': Array(0.14583334, dtype=float32)}
Epoch 3 | Train: {'loss': Array(2.0661802, dtype=float32), 'accuracy': Array(0.1328125, dtype=float32)} | Val: {'loss': Array(2.0704682, dtype=float32), 'accuracy': Array(0.125, dtype=float32)}
Epoch 4 | Train: {'loss': Array(2.0602567, dtype=float32), 'accuracy': Array(0.16210938, dtype=float32)} | Val: {'loss': Array(2.0706441, dtype=float32), 'accuracy': Arr

In [37]:


def evaluate_test_set(model, X_test_vram, y_test_vram, batch_size=32):
    """
    Evaluates the trained model on the held-out test set directly from VRAM.
    """
    test_metrics = nnx.MultiMetric(
        loss=nnx.metrics.Average('loss'),
        accuracy=nnx.metrics.Accuracy()
    )
    
    # Lock model state (disables dropout, uses running batch norm stats)
    model.eval()
    
    num_samples = len(y_test_vram)
    num_batches = math.ceil(num_samples / batch_size)
    
    print(f"Evaluating {num_samples} test samples across {num_batches} batches...")
    
    for i in range(num_batches):
        # Calculate bounds for the current slice
        start_idx = i * batch_size
        end_idx = min((i + 1) * batch_size, num_samples)
        
        # Slice directly from VRAM; no Host-to-Device transfer occurs
        batch = {
            "mel": X_test_vram[start_idx:end_idx],
            "label": y_test_vram[start_idx:end_idx]
        }
        
        # Note: The final step will trigger a brief XLA recompilation 
        # because the batch dimension will be smaller than 32.
        eval_step(model, test_metrics, batch)
        
    results = test_metrics.compute()
    final_loss = results['loss'].item()
    final_acc = results['accuracy'].item()
    
    print("-" * 30)
    print("Held-Out Test Set Results:")
    print(f"Loss:     {final_loss:.4f}")
    print(f"Accuracy: {final_acc:.4f}")
    print("-" * 30)
    
    return final_loss, final_acc

In [38]:

print("\nEvaluating on Held-Out Test Set...")
text_model.eval()

test_metrics = nnx.MultiMetric(
    loss=nnx.metrics.Average('loss'),
    accuracy=nnx.metrics.Accuracy()
)

num_test_samples = len(y_test_vram)
num_test_batches = math.ceil(num_test_samples / BATCH_SIZE)

for i in range(num_test_batches):
    start_idx = i * BATCH_SIZE
    end_idx = min((i + 1) * BATCH_SIZE, num_test_samples)
    
    test_batch = {
        "text": X_text_test_vram[start_idx:end_idx],
        "label": y_test_vram[start_idx:end_idx]
    }
    
    text_eval_step(txt_model, test_metrics, test_batch)

test_results = test_metrics.compute()
print("-" * 30)
print("TextRNN Test Results:")
print(f"Loss:     {test_results['loss']:.4f}")
print(f"Accuracy: {test_results['accuracy']:.4f}")
print("-" * 30)


Evaluating on Held-Out Test Set...


NameError: name 'text_model' is not defined

In [ ]:
# Execute after the final epoch
test_loss, test_accuracy = evaluate_test_set(
    model, 
    X_test_vram, 
    y_test_vram, 
    batch_size=32
)